# 🧠 Stage 2: Unified Baseline Model Evaluation & Collapse Probing
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs — AI Research Division  
**Authors:** Omar Abdelhamid, Nour Walid (Equal Contribution)  
**Supervisor:** Dr. Ghada Soliman  

---

### 🎯 Objectives of this Unified Evaluation Suite:
1. **Model Loading:** Load **`Qwen/Qwen2.5-Coder-1.5B-Instruct`** in **4-bit NF4 precision** (fits comfortably within 4 GB VRAM on our RTX 3070 Ti).
2. **Reduction Ladder Evaluation:** Execute deterministic **Pass@1** ($T=0.0$) across all six levels (**L0 to L5**).
3. **Diagnostic Error Taxonomy:** Automatically categorize all failures into `on_path`, `off_path`, `wrong_template`, `syntax_error`, or `runtime_error`.
4. **Metric Suite Calculation:** Quantify the baseline **Collapse Point** ($\ell^*$), **Ladder AUC** ($\mathcal{A}$), **Consistency Delta** ($\Delta_c$), and **Memorization Risk Index** (MRI).
5. **Artifact Generation:** Automatically export JSON reports and publication-ready degradation curves to `results/baseline/`.

## 1. System Environment & GPU VRAM Verification

In [ ]:
import os
import sys
import torch
from IPython.display import Image, display

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.evaluation import EvaluationEngine

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Initial VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")

print("\n✅ Unified EvaluationEngine ready!")

## 2. Initialize Unified Evaluation Engine (4-bit NF4 Model & Sandbox)

In [ ]:
engine = EvaluationEngine(
    model_name="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    data_cache_dir="../data/ladder",
    results_dir="../results"
)

if torch.cuda.is_available():
    print(f"\n🔥 Post-load VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB (Fits comfortably in 8GB budget!)")

## 3. Run Full Reduction Ladder Evaluation (L0 to L5)
Evaluates the un-tuned baseline model across all 664 problems and diagnoses failure categories in real-time.

In [ ]:
# Execute evaluation across all levels L0 -> L5
suite_report = engine.run_full_ladder(
    output_tag="baseline",
    evaluate_pass5=False  # Set to True for exploration sampling
)

## 4. Multi-Dimensional Summary Metrics Table

In [ ]:
summary_df = engine.get_summary_dataframe()
print("\n🏆 Model Evaluation Summary Matrix:")
display(summary_df)

## 5. Visualizations: Baseline Degradation Curve & Error Taxonomy Distribution

In [ ]:
print("📈 1. Pass@1 Degradation Curve Across Transformations:")
display(Image("../results/baseline/baseline_degradation_curve.png"))

print("\n📊 2. Diagnostic Error Taxonomy (On-Path vs Off-Path vs Wrong-Template):")
display(Image("../results/baseline/baseline_error_taxonomy.png"))

## 6. Qualitative Error Inspection: Investigating Template Collapse Cases

In [ ]:
for lvl in ["L1", "L2", "L3"]:
    if lvl in suite_report.level_reports:
        rep = suite_report.level_reports[lvl]
        failures = [t for t in rep.task_results if not t["p1_passed"]]
        if failures:
            sample = failures[0]
            print(f"\n{'='*35} Sample Failure in {lvl} (Task ID: {sample['task_id']}) {'='*35}")
            print(f"Diagnosis: {sample['p1_error_category']}")
            print(f"Error Message: {sample['p1_error_message'][:200]}")
            print("Generated Code Snippet:")
            print(sample["p1_code"][:300] + "...")